# 03 · Vision-language backbone


> **Research prototype — not a medical device.** Nothing produced by these notebooks may be
> used to diagnose, treat, or make any decision about a patient.


## Architecture

```
film  ──► BiomedCLIP ViT-B/16 ──► 197 patch tokens ─┐
                                                    ├─► bidirectional cross-attention
report ─► PubMedBERT ──────────► 256 text tokens ───┘   (2 blocks) ─► gated pool
                                                                        │
                                                                   node feature (384-d)
                                                                        │
                                                            ┌───────────┴───────────┐
                                                        4 logits            (stage 3: hypergraph)
```

BiomedCLIP is pretrained on 15M biomedical image–text pairs, so the two towers
already share a space — the fusion trunk does not have to learn alignment from
12k studies.

## What is trained

Only the **last 4 ViT blocks** and the **last 2 BERT blocks**, plus all norms,
the projections, the fusion trunk and the heads. Full fine-tuning of both towers
overfits this subset and will not fit a T4 at a useful batch size.

## Two details that matter

**No horizontal flip.** Chest anatomy is not left–right symmetric; flipping a
film invents dextrocardia and destroys the cue the Cardiomegaly head needs.

**Modality dropout (15%).** The report is blanked on 15% of training steps, so
the model stays usable when the demo gets an image with no report. Without it,
an image-only request falls off a cliff.

In [ ]:
import subprocess, sys
print(sys.version)
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip() or "no GPU reported")
except FileNotFoundError:
    print("nvidia-smi not found - you are on CPU. Runtime > Change runtime type > T4 GPU.")
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# --- 1. where results live -------------------------------------------------
# Mounting Drive is strongly recommended: Colab disconnects, and every stage
# here writes a resumable checkpoint. Without Drive you start over.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = '/content/drive/MyDrive/dvlhg'
else:
    RUN_ROOT = '/content/dvlhg'

# --- 2. get the code -------------------------------------------------------
# Pick ONE. 'clone' is easiest once you have pushed this repo to GitHub.
SOURCE = 'clone'        # 'clone' | 'zip' | 'drive'
REPO_URL = 'https://github.com/abelsangeeth/DVL-Hyperparameter-for-Lung-Disease-Diagnosis.git'
ZIP_PATH = '/content/dvl-hypergraph.zip'          # if SOURCE == 'zip'
DRIVE_CODE = '/content/drive/MyDrive/dvl-hypergraph'  # if SOURCE == 'drive'

import os, shutil, subprocess, sys
CODE = '/content/dvl-hypergraph'
if not os.path.exists(CODE):
    if SOURCE == 'clone':
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, CODE], check=True)
    elif SOURCE == 'zip':
        if not os.path.exists(ZIP_PATH):
            from google.colab import files
            up = files.upload()            # choose the zip from scripts/make_colab_zip.py
            ZIP_PATH = '/content/' + next(iter(up))
        shutil.unpack_archive(ZIP_PATH, '/content/')
    elif SOURCE == 'drive':
        shutil.copytree(DRIVE_CODE, CODE)
print('code at', CODE, '| contents:', sorted(os.listdir(CODE))[:8])

# --- 3. dependencies -------------------------------------------------------
# Colab already ships torch/torchvision built for its CUDA - never reinstall them.
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'open_clip_torch>=2.24', 'timm>=0.9.12', 'transformers>=4.35',
                'fastapi', 'uvicorn', 'python-multipart'], check=True)

sys.path.insert(0, os.path.join(CODE, 'src'))
os.chdir(CODE)
os.environ['PYTHONPATH'] = os.path.join(CODE, 'src')
os.environ['RUN_ROOT'] = RUN_ROOT
print('run root ->', RUN_ROOT)

In [ ]:
# Everything below is overridden on the command line, so this cell is the only
# place you need to edit. Values here are tuned for a Colab T4.
CFG = dict(
    subset_size = 12000,     # frontal studies pulled from MIMIC-CXR-JPG
    text_mode   = 'indication',   # see docs/LEAKAGE.md before changing this
    batch_size  = 24,
    epochs      = 12,
    diffusion_epochs = 30,
    synth_per_class  = 500,
)

def dvlhg(command, **overrides):
    """Run a dvlhg subcommand with RUN_ROOT and any overrides applied."""
    import os, shlex, subprocess, sys
    args = [sys.executable, '-m', 'dvlhg.cli', *shlex.split(command)]
    args += ['--set', f"paths.root={os.environ['RUN_ROOT']}"]
    for key, value in overrides.items():
        args += ['--set', f'{key}={value}']
    print('$', ' '.join(args[2:]))
    return subprocess.run(args, check=True)

In [ ]:
SOURCE_DATASET = 'mimic'

### Train

~8–15 minutes per epoch on a T4 at batch 24. **Every epoch writes a full
checkpoint** (weights, optimiser, scheduler, scaler, history) — if Colab drops,
re-run this cell and it resumes from the last completed epoch.

If you hit CUDA OOM: lower `train.batch_size` to 16 and raise
`train.accum_steps` to 2 (same effective batch), or set
`vlm.unfreeze_vision_blocks=2`.

In [ ]:
dvlhg('train',
      **{'data.source': SOURCE_DATASET,
         'text.mode': CFG['text_mode'],
         'train.epochs': CFG['epochs'],
         'train.batch_size': CFG['batch_size']})

### Training curves

In [ ]:
import json, os, matplotlib.pyplot as plt
history = json.load(open(os.path.join(os.environ['RUN_ROOT'], 'logs', 'backbone_history.json')))
epochs = [h['epoch'] for h in history]

fig, axes = plt.subplots(2, 1, figsize=(6, 5), sharex=True)
axes[0].plot(epochs, [h['train_loss'] for h in history], color='#2a78d6', lw=2)
axes[0].set_ylabel('train loss'); axes[0].grid(color='#e6e5e1')
axes[1].plot(epochs, [h['val_auroc_macro'] for h in history], color='#2a78d6', lw=2,
             marker='o', ms=4, label='val macro AUROC')
axes[1].plot(epochs, [h['val_f1_macro'] for h in history], color='#eb6834', lw=2,
             marker='s', ms=4, ls='--', label='val macro F1')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('score'); axes[1].grid(color='#e6e5e1')
axes[1].legend(frameon=False)
for ax in axes:
    for side in ('top', 'right'): ax.spines[side].set_visible(False)
plt.tight_layout(); plt.show()

best = max(history, key=lambda h: h['val_auroc_macro'])
print(f"best epoch {best['epoch']}: val macro AUROC {best['val_auroc_macro']:.4f}")

**Reading the curves.** Validation AUROC plateauing while train loss keeps
falling is normal and the early-stopping rule handles it. Validation AUROC
*falling* for several epochs means the encoder learning rate is too high —
lower `train.encoder_lr_scale` from 0.1 to 0.05.